# 01 | Homicídios do CDC

Vamos baixar os registros de 2019 a 2024 e verificar se cada um dos 50 estados aparece em todos os anos.

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Baixar

Buscamos homicídios **totais** na fonte oficial. Não são apenas os casos com armas de fogo.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import requests
import pandas as pd
from io import StringIO
url = "https://data.cdc.gov/resource/fpsi-y8tj.csv"
params = {"$where": "intent='All_Homicide' AND period IN ('2019','2020','2021','2022','2023','2024')", "$limit": 1000}
resposta = requests.get(url, params=params, timeout=40)
resposta.raise_for_status()
cdc_raw = pd.read_csv(StringIO(resposta.text))
print("Linhas recebidas:", len(cdc_raw))

## Bloco 2: Conferir

Retiramos Washington, D.C. porque o estudo usa somente os 50 estados.

In [ ]:
estados = cdc_raw[cdc_raw["name"] != "District of Columbia"].copy()
assert len(estados) == 300, "Esperávamos 50 estados em seis anos."
assert estados["name"].nunique() == 50
assert not estados.duplicated(["name", "period"]).any()
assert estados[["rate", "count_sup"]].notna().all().all()
print("OK: 300 registros, 50 estados, sem duplicações ou taxas vazias.")
display(estados.groupby("period")["name"].nunique())

## Bloco 3: Salvar e observar

Guardamos o CSV original para que os próximos notebooks possam usá-lo. A média mostrada é a **média entre estados**, não a taxa nacional.

In [ ]:
destino = ROOT / "data/raw/cdc_homicides_2019_2024.csv"
destino.parent.mkdir(parents=True, exist_ok=True)
cdc_raw.to_csv(destino, index=False)
print("Salvo:", destino.relative_to(ROOT))
display(estados.groupby("period")["rate"].mean().round(2))